In [18]:
import torch 
import torch.nn as nn

In [19]:
class NoisyTopKGating(nn.Module):
    def __init__(self, D, N, K):
        super(NoisyTopKGating, self).__init__()
        self.W_G = torch.nn.Parameter(torch.randn((D, N)))
        self.W_N = torch.nn.Parameter(torch.randn((D, N)))
        self.normal_dist = torch.distributions.Normal(loc=0, scale=1)
        self.softplus = nn.Softplus()

        self.D = D
        self.N = N 
        self.K = K


    def forward(self, X:torch.tensor): # (B, S, D)
        (B, S, D) = X.shape
        K, N = self.K, self.N
        W_G = X @ self.W_G # (B, S, D) @ (D, N) = (B, S, N)
        W_N = X @ self.W_N

        assert W_G.shape == (B, S, self.N)
        
        e = self.normal_dist.sample((B, S, self.N)) 

        H = W_G + e * self.softplus(W_N) # (B, S, N)
        KV, KI = torch.topk(H, k=self.K, dim=-1) # (B, S, K)

        assert KI.shape == (B, S, self.K)
        assert KV.shape == (B, S, self.K)


        G = nn.Softmax(dim=-1)(KV)

        assert G.shape == (B, S, K)

        # construct G_N (B, S, N) from G (B, S, K) and KI (B, S, N) where KI are indices. Torch.scatter will help here. 
        G_N = torch.zeros((B, S, N), dtype=G.dtype)
        G_N.scatter_(dim=2, index=KI, src=G)

        assert G_N.shape == (B, S, N)

        probs = G_N.mean(dim=(0, 1))

        assert probs.shape == (self.N, )

        threshold_logit = KV[:,:,-1:]

        assert threshold_logit.shape == (B, S, 1)

        D = (W_G - threshold_logit) / self.softplus(W_N)

        assert D.shape == (B, S, self.N)

        f = self.normal_dist.cdf(D).mean(dim=(0, 1))

        assert f.shape == (self.N, )

        aux_loss = torch.sum(f * probs)

        return G, aux_loss, KI     



        



noisy_top_k_gating = NoisyTopKGating(D=2, N=5, K=3)

X = torch.randn((1, 4, 2)).float()
noisy_top_k_gating.forward(X)

(tensor([[[0.4154, 0.3959, 0.1887],
          [0.4822, 0.2670, 0.2507],
          [0.5751, 0.2154, 0.2095],
          [0.5351, 0.3030, 0.1619]]], grad_fn=<SoftmaxBackward0>),
 tensor(0.5964, grad_fn=<SumBackward0>),
 tensor([[[0, 4, 1],
          [4, 3, 0],
          [0, 1, 4],
          [4, 0, 3]]]))

In [20]:
a = torch.randn((1, 1, 4, 3))
print(a)
print(a.shape)

b = torch.tensor([
    [1, 2],
    [2, 0],
    [0, 1],
    [1, 0]
    ]).unsqueeze(0).unsqueeze(0)

print(b.shape)
c = torch.gather(a, dim=2, index=b)

print(c)
print(c.shape)

tensor([[[[ 0.5104, -0.3107,  1.1037],
          [ 1.7647, -0.4287,  0.6959],
          [ 0.5847,  1.5733,  0.9362],
          [ 0.1881,  0.3991, -0.0859]]]])
torch.Size([1, 1, 4, 3])
torch.Size([1, 1, 4, 2])
tensor([[[[ 1.7647,  1.5733],
          [ 0.5847, -0.3107],
          [ 0.5104, -0.4287],
          [ 1.7647, -0.3107]]]])
torch.Size([1, 1, 4, 2])


In [21]:
src = torch.arange(1, 11).reshape((2, 5))
print(src)

index = torch.tensor([[0, 1, 2, 0]])

print(index, index.shape)

torch.zeros((3, 5), dtype=src.dtype).scatter_(0, index, src)

tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])
tensor([[0, 1, 2, 0]]) torch.Size([1, 4])


tensor([[1, 0, 0, 4, 0],
        [0, 2, 0, 0, 0],
        [0, 0, 3, 0, 0]])

In [22]:
x = torch.randn((2, 3, 4))
y = torch.randn((4, 5))
z = x @ y
z.shape

a = torch.randn((4, 5))
b = a.expand(2, -1, -1)
print(a.shape, b.shape)

torch.Size([4, 5]) torch.Size([2, 4, 5])


In [23]:
class Block(nn.Module):
    def __init__(self, D):
        super(Block, self).__init__()
        self.a1 = nn.Linear(D, D)
        self.a2 = nn.Linear(D, D)

    def forward(self, x):
        h1 = self.a1(x)
        h2 = nn.GELU()(h1)
        return self.a2(h2)

class ShazeerMOE(nn.Module):
    def __init__(self, D, N, K):
        super(ShazeerMOE, self).__init__()
        self.D = D
        self.N = N 
        self.K = K
        
        self.experts = nn.Parameter(torch.randn((N, D, D), requires_grad=True))

        self.noisy_gating = NoisyTopKGating(D, N, K)

    def forward(self, X):
        B, S, D = X.shape
        M = B * S
        K, N = self.K, self.N

        G, aux_loss, KI = self.noisy_gating(X)
        X = X.reshape((M, D))

        G = G.reshape((-1, K))

        assert G.shape == (M, K)

        KI = KI.reshape((-1, K))

        assert KI.shape == (M, K)

        EK = self.experts[KI]

        # N gets replaced with M, K 

        assert EK.shape == (M, K, D, D)

        XK = X.unsqueeze(dim=1).expand(-1, K, D)

        assert XK.shape == (M, K, D)

        Y = torch.einsum("mkd,mkde,mk->me", XK, EK, G)

        assert Y.shape == (M, D)

        return Y, aux_loss

        


D = 2
N = 4
K = 2
B = 1
S = 3
shazeer_moe = ShazeerMOE(D, N, K)
a = torch.randn((B, S, D))
shazeer_moe.forward(a)


(tensor([[-0.8879, -0.8253],
         [-1.0735, -0.4003],
         [-0.4313,  3.0296]], grad_fn=<ViewBackward0>),
 tensor(0.5230, grad_fn=<SumBackward0>))

In [26]:
src = torch.arange(1, 11).reshape((2, 5))
src
index = torch.tensor([[0, 1, 2, 0]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(0, index, src)
index = torch.tensor([[0, 1, 2], [0, 1, 4]])
torch.zeros(3, 5, dtype=src.dtype).scatter_(1, index, src)

tensor([[1, 2, 3, 0, 0],
        [6, 7, 0, 0, 8],
        [0, 0, 0, 0, 0]])